# 04 — Krigeage et simulation (Turning Bands)

## Objectif de ce notebook

Créer une grille d'estimation, réaliser le krigeage de **Y.Pb** selon trois méthodes (Simple, Ordinaire, Universel), puis réaliser des
simulations conditionnelles par la méthode des **bandes tournantes** (Turning Bands Simulation).

##  Prérequis

Ce notebook reprend le contexte construit dans les notebooks 02 et 03 : base `db` avec la variable transformée **Y.Pb**, et le modèle de
variogramme ajusté `model_final` (Nugget + Sphérique, nugget=0.21,portée=0.60 km).

## Contenu

1. Création de la grille d'estimation (équivalent au voxet SKUA-Gocad)
2. Krigeage Simple (SK)
3. Krigeage Ordinaire (OK)
4. Krigeage Universel (UK)
5. Widget interactif de visualisation des cartes de krigeage
6. Simulation par bandes tournantes (TBS) — 4 réalisations

## Sortie de ce notebook

Les cartes de krigeage et de simulation générées sur la grille `grid` peuvent être exportées dans `outputs/` pour analyse ultérieure.

## Rappel : recréer le contexte (notebooks 02 et 03)

Cette cellule reproduit rapidement les étapes essentielles des notebooks précédents afin que ce notebook puisse être exécuté de façon autonome.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gstlearn as gl
import gstlearn.plot as gp
import ipywidgets as widgets
from ipywidgets import interactive_output

# Ajout du chemin d'importation pour le module geostat_utils (remonte de jura/ jusqu'à src/)
sys.path.append(os.path.abspath(os.path.join("..", "..")))
import geostat_utils as gutils

# Reconstruction du contexte à l'aide de la fonction d'initialisation du module
data_jura, db, anam_hermite = gutils.preparer_db_gaussienne()

nombre_classe = 17
taille_classe = 0.291066
params_vario_omni = gl.VarioParam.createOmniDirection(
    nlag=nombre_classe, dlag=taille_classe, toldis=0.1
)
vario_exp_omni = gl.Vario(params_vario_omni)
err = vario_exp_omni.compute(db)

# ---- Modèle final retenu : Nugget + Sphérique (notebook 03) ----
model_final = gl.Model()
model_final.addCovFromParam(gl.ECov.NUGGET, sill=0.21 * 0.991462)
model_final.addCovFromParam(gl.ECov.SPHERICAL, range=0.60, sill=(1 - 0.21) * 0.991462)


## 1. Création de la grille d'estimation

Une grille régulière de **100 × 100 cellules** est créée pour couvrir l'ensemble du domaine d'étude (équivalent au voxet dans SKUA-Gocad) :

- **NI** : 100 cellules selon X
- **NJ** : 100 cellules selon Y
- **Maille selon X** : 0.043 km
- **Maille selon Y** : 0.051 km
- **Total** : 10 000 cellules

In [ ]:
# ---- Création de la grille ----
grid = gutils.creer_grille(data_jura)
print(grid)


## 2. Krigeage (SK, OK, UK) 

Le krigeage est appliqué sur la grille `grid` avec la variable **Y.Pb**. Le modèle utilisé est le modèle **Nugget + Sphérique** ajusté précédemment (notebook 03).Aussi le Widget interactif d'affichage des cartes de krigeage permet de sélectionner la méthode (SK, OK, UK) et le type de carte (estimation ou variance/écart-type).

Trois méthodes sont utilisées :

- **Krigeage Simple (SK)** : la moyenne est connue et fixe (= moyenne globale de Y.Pb)
- **Krigeage Ordinaire (OK)** : la moyenne est inconnue et estimée localement
- **Krigeage Universel (UK)** : la moyenne est modélisée par une tendance linéaire (drift d'ordre 1), plus flexible que SK et OK car il modélise une tendance spatiale

In [7]:
# ---- Krigeage Simple (SK) ----
# La moyenne est connue et fixe = moyenne globale de Y.Pb

voisin_unique = gl.NeighUnique.create()
mean_ypb = np.mean(db.getColumn("Y.Pb"))
print(f"moyenne Y.Pb = {mean_ypb:.3f}")

# ---- Modèle SK ----

model_SK = model_final.clone()
model_SK.setMean(mean_ypb)

gl.kriging(
    dbin=db,
    dbout=grid,
    model=model_SK,
    neigh=voisin_unique,
    flag_est=True,
    flag_varz=True,
    namconv=gl.NamingConvention("Kriging_simple")
)

# ---- Krigeage Ordinaire (OK) ----
# La moyenne est inconnue et estimée localement avec une dérive constante

model_OK = model_final.clone()
model_OK.addDrift(gl.DriftM())

gl.kriging(
    dbin=db,
    dbout=grid,
    model=model_OK,
    neigh=voisin_unique,
    flag_est=True,
    flag_varz=True,
    namconv=gl.NamingConvention("Kriging_ordinaire")
)
# ---- Krigeage Universel (UK) ----
# La moyenne est modélisée par une tendance linéaire (drift d'ordre 1)
# Plus flexible que SK et OK car il modélise une tendance spatiale

model_UK = model_final.clone()
model_UK.setDriftIRF(order=1)   # tendance linéaire pour UK

# ---- Calcul du krigeage universel ----
gl.kriging(
    dbin=db,
    dbout=grid,
    model=model_UK,
    neigh=voisin_unique,
    flag_est=True,
    flag_std=True,
    flag_varz=False,
    namconv=gl.NamingConvention("Kriging_universel")
)
#-----utilisation widegt interactif pour l'affichage des cartes de krigeage----

widget_krigeage = widgets.Dropdown(
    options=[
    ('Krigeage Simple (SK)' , 'simple'),
    ('Krigeage Ordinaire (OK)' , 'ordinaire'),
    ('Krigeage Universel (UK)' , 'universel')
],
value='simple',
description='Méthode'
)
widget_carte_affichage = widgets.Dropdown(
    options=[
        ('Estimation', 'estim'),
        ('Variance/Écart-type','varz')
    ],
    value='estim',
    description='Carte:'
)
def afficher_krigeage(choix_methode,choix_carte):
    if choix_methode == 'universel' and choix_carte == 'varz':
        nom_carte ='Kriging_universel.Y.Pb.stdev'
    else:
        nom_carte = f'Kriging_{choix_methode}.Y.Pb.{choix_carte}'

    fig, ax = gp.init(figsize=(8, 8), flagEqual=True)
    ax.raster(grid, name=nom_carte, flagLegend=True, cmap="jet")
    ax.symbol(db, c="black")
    ax.decoration(title=f"Krigeage {choix_methode} — {choix_carte}")
    plt.show()

#------Affichage du widegt-krigeage----

display(widgets.VBox([
    widget_krigeage,
    widget_carte_affichage,
    interactive_output(afficher_krigeage, {
        'choix_methode': widget_krigeage,
        'choix_carte':    widget_carte_affichage
    })
]))

moyenne Y.Pb = 0.000


## 3. Turning Bands Simulation de Pb

La moyenne de `Y.Pb = 0.00` confirme que la transformation Normal Score est correcte.
On utilise la fonction `gl.simtub()` avec **4 réalisations**, **1000 bandes** et `seed=12345` pour la reproductibilité. Le modèle final (Nugget + Sphérique) est cloné et sa moyenne est fixée à `mean_ypb`.

Les 4 simulations sont affichées dans l'espace gaussien (vmin=-3, vmax=3).Une **back-transformation** serait nécessaire pour revenir aux valeurs originales de Pb (voir notebook 02 pour la méthode de back-transformation).

In [8]:
#---- Turning Bands Simulation de Pb ----
mean_ypb = np.mean(db.getColumn("Y.Pb"))
print(f"Moyenne de Y.Pb = {mean_ypb:.2f}")
      
# Modèle de simulation
model_simu_pb = model_final.clone() 
model_simu_pb.setMean(mean_ypb)

nb_simulations = 4
nb_bandes = 1000
seed = 12345

gl.simtub(
    dbin=db,
    dbout=grid,
    model=model_simu_pb,
    neigh=voisin_unique,
    nbsimu=nb_simulations,
    nbtuba=nb_bandes,
    seed=seed,
    namconv=gl.NamingConvention("YPbSimu")
)

#-----Widget-affichege_des quatres realisations avec la methode de TBS
widget_realisation = widgets.ToggleButtons(
    options=[
        ('Réalisation 1', 1),
        ('Réalisation 2', 2),
        ('Réalisation 3', 3),
        ('Réalisation 4', 4)
    ],
    value=1,
    description='Cartes (TBS):',
    button_style='warning'  
)
def afficher_realisation (numero):
    fig, ax = gp.init (figsize=(8, 8), flagEqual = True)
    ax.raster(grid, name=f"YPbSimu.Y.Pb.{numero}",flagLegend=True, cmap="jet", vmin=-3, vmax=3)
    ax.symbol(db, c="black")
    ax.decoration(title=f"Réalisation conditionnelle n°{numero} de Y.Pb — Simulation \n par Bandes Tournantes (TBS)")
    plt.show()

display (widgets.VBox([ widget_realisation,
    interactive_output(afficher_realisation, {'numero': widget_realisation
    })
]))

Moyenne de Y.Pb = 0.00


## Conclusion

Le krigeage (SK, OK, UK) et la simulation par bandes tournantes (TBS) ont été réalisés sur la grille 100×100 avec le modèle Nugget + Sphérique. Les cartes obtenues permettent de visualiser l'estimation spatiale du Plomb (Pb) ainsi que l'incertitude associée (variance) et plusieurs scénarios équiprobables (simulations).